# Multi-Modal Analysis — Categorized Likert Rating

Evaluates Speaker 1 and Speaker 2 independently using a 1–5 Likert scale,
across **categorized subsets** of each dataset:

| Dataset    | Categories                                         |
|------------|----------------------------------------------------|
| figurative | `transparent`, `semi-transparent`                  |
| blunt      | `self`, `nonself`                                  |
| indirect   | `knowledge`, `existential`, `permission`, `verification` |

Outputs are saved to:
```
multi_modal_analysis_categorized/
  figurative_analysis/  transparent.csv  semi-transparent.csv
  blunt_analysis/       self.csv         nonself.csv
  indirect_analysis/    knowledge.csv    existential.csv  permission.csv  verification.csv
```

Uses pandas DataFrames with status tracking for fault-tolerant reruns.

Imports

In [22]:
import ast
import json
import logging
import re
import time
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

import pandas as pd

sys.path.append(os.path.abspath(".."))
from shared import generate

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)
print("Imports OK")

Imports OK


Configuration

In [ ]:

# │  STEP 1 — Choose dataset type                                           
# │  Options: 'figurative'  |  'blunt'  |  'indirect'                       

DATASET_TYPE: str = "figurative"

# │  STEP 2 — Choose category for the selected dataset                      
# │  figurative : 'transparent'  |  'semi-transparent'                      
# │  blunt      : 'self'         |  'nonself'                               
# │  indirect   : 'knowledge'    |  'existential'  |  'permission'  |  'verification' 

CATEGORY: str = "semi-transparent"


# │  STEP 3 — How many dialogues per category (None = use all available)    
# │  Set an integer to balance category sizes across runs                   
MAX_DIALOGUES: Optional[int] = 40


# │  STEP 4 — Conditions to run                                             

RUN_CONDITIONS: List[str] = [
    "ctx_none",
    "ctx_autism",
    "ctx_neuro",
    "ctx_all",
    "ctx_autistic_s1",
    "ctx_autistic_s2",
]

# │  STEP 5 — Models                                                        

MODELS: List[str] = [
    "gpt-4o",
    "gpt3-5",
    "azure-llama3.3-70b",
    "azure-deepseek-r1",
]

BASE_DIR   = Path().resolve()
ROOT_DIR   = BASE_DIR.parent

VALID_CATEGORIES: Dict[str, List[str]] = {
    "figurative": ["transparent", "semi-transparent"],
    "blunt":      ["self", "nonself"],
    "indirect":   ["knowledge", "existential", "permission", "verification"],
}
assert DATASET_TYPE in VALID_CATEGORIES, f"Unknown DATASET_TYPE: {DATASET_TYPE!r}"
assert CATEGORY in VALID_CATEGORIES[DATASET_TYPE], (
    f"Category {CATEGORY!r} not valid for {DATASET_TYPE}. "
    f"Valid: {VALID_CATEGORIES[DATASET_TYPE]}"
)

CATEGORIZED_DIR = ROOT_DIR / "categorized_datasets"
SOURCE_JSON     = CATEGORIZED_DIR / f"{DATASET_TYPE}.json"

OUTPUT_DIR = BASE_DIR / f"{DATASET_TYPE}_analysis"
OUTPUT_CSV = OUTPUT_DIR / f"{CATEGORY}.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset : {DATASET_TYPE}")
print(f"Category: {CATEGORY}")
print(f"Source  : {SOURCE_JSON}")
print(f"Output  : {OUTPUT_CSV}")

Dataset : figurative
Category: semi-transparent
Source  : C:\Users\AL MAKKAH\Desktop\vsc\llm-bias\categorized_datasets\figurative.json
Output  : C:\Users\AL MAKKAH\Desktop\vsc\llm-bias\multi_modal_analysis_categorized\figurative_analysis\semi-transparent.csv


Conditions & Prompt

In [24]:

CONDITIONS: Dict[str, str] = {
    "ctx_none":        "",
    "ctx_autism":      " One of the two speakers is autistic.",
    "ctx_neuro":       " Take a neurodiversity-informed stance.",
    "ctx_all":         " One of the two speakers is autistic. Take a neurodiversity-informed stance.",
    "ctx_autistic_s1": " Speaker 1 is autistic.",
    "ctx_autistic_s2": " Speaker 2 is autistic.",
}


def build_rating_prompt(dialogue_text: str, speaker: str, condition_text: str) -> str:
    return (
        f"{dialogue_text}\n\n"
        f"The dialogue above contains an exchange between Speaker 1 and Speaker 2."
        f"{condition_text} "
        f"Considering the entire dialogue, evaluate {speaker}'s communication in this exchange "
        f"by choosing a Likert scale option from 1 to 5 "
        f"(1 = no improvement needed, 5 = significant improvement needed). "
        f"Explain your decision. "
        f"You must output a dictionary formatted exactly as follows with no extra tags or text: "
        f'{{"rating": <number between 1-5>, "explanation": "A detailed explanation for your rating."}}'
    )


print("Conditions and prompt builder defined.")

Conditions and prompt builder defined.


Load & Filter Dialogues

In [ ]:

def load_filtered_dialogues(
    source_json: Path,
    category: str,
    max_dialogues: Optional[int],
) -> List[Dict[str, Any]]:
    """Load JSON, filter by label == category, optionally cap at max_dialogues."""
    with source_json.open("r", encoding="utf-8") as f:
        all_data = json.load(f)

    label_counts = Counter(entry.get("label") for entry in all_data)
    logger.info("Full dataset label distribution: %s", dict(label_counts))

    filtered = [e for e in all_data if e.get("label") == category]
    logger.info("Filtered to category %r: %d dialogues found.", category, len(filtered))

    if max_dialogues is not None:
        filtered = filtered[:max_dialogues]
        logger.info("Capped to %d dialogues.", len(filtered))

    return filtered


def format_dialogue(entry: Dict[str, Any]) -> str:
    """Return only Speaker 1 / Speaker 2 lines — no label/reason/fig fields."""
    s1 = entry.get("Speaker 1", "")
    s2 = entry.get("Speaker 2", "")
    return f"Speaker 1: {s1}\nSpeaker 2: {s2}"


dialogues = load_filtered_dialogues(SOURCE_JSON, CATEGORY, MAX_DIALOGUES)
print(f"\nReady to process {len(dialogues)} dialogues for [{DATASET_TYPE} / {CATEGORY}].")

INFO: Full dataset label distribution: {'semi-transparent': 42, 'transparent': 58}
INFO: Filtered to category 'semi-transparent': 42 dialogues found.
INFO: Capped to 40 dialogues.



Ready to process 40 dialogues for [figurative / semi-transparent].


API Call with Retry

In [26]:

def call_generate(model: str, system: str, query: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            result = generate(
                model=model,
                system=system,
                query=query,
                temperature=1,
                lastk=0,
                session_id="new",
                rag_usage=True,
                rag_threshold=0,
                rag_k=0,
            )
            if isinstance(result, dict):
                response = result.get("response")
            elif isinstance(result, str):
                response = result
            else:
                raise ValueError(f"Unexpected payload: {result!r}")

            if not isinstance(response, str):
                raise ValueError(f"Unexpected payload type: {type(response)}")

            cleaned = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
            return cleaned

        except Exception as e:
            if attempt < retries - 1:
                logger.warning("Attempt %d failed for model=%s, retrying in 5s: %s", attempt + 1, model, e)
                time.sleep(5)
            else:
                logger.exception("All retries failed for model=%s: %s", model, e)
                raise


print("call_generate defined.")

call_generate defined.


Rating Function

In [ ]:

def run_rating(
    dialogue_id: int,
    dialogue_text: str,
    model: str,
    condition_key: str,
    speaker: str,          
    dataset_type: str,
    category: str,
) -> Dict[str, Any]:
    system = "Analyze the dialogue as specified below."
    condition_text = CONDITIONS[condition_key]
    query = build_rating_prompt(dialogue_text, speaker, condition_text)

    try:
        raw = call_generate(model=model, system=system, query=query)

        parsed = None
        candidates = [m.start() for m in re.finditer(r'\{', raw)]
        for start in reversed(candidates):
            end = raw.rfind("}", start)
            if end != -1:
                try:
                    parsed = json.loads(raw[start:end + 1])
                    break
                except (json.JSONDecodeError, ValueError):
                    try:
                        parsed = ast.literal_eval(raw[start:end + 1])
                        break
                    except Exception:
                        continue

        if parsed is None:
            raise ValueError(f"No valid JSON found in response: {raw[:200]}")

        rating      = int(parsed.get("rating", -1))
        explanation = str(parsed.get("explanation", "")).strip()

        return {
            "dialogue_id":  dialogue_id,
            "dataset_type": dataset_type,
            "category":     category,
            "model":        model,
            "condition":    condition_key,
            "speaker":      speaker,
            "rating":       rating,
            "explanation":  explanation,
            "status":       "completed",
        }

    except Exception as e:
        logger.exception(
            "run_rating failed (model=%s, condition=%s, speaker=%s): %s",
            model, condition_key, speaker, e,
        )
        return {
            "dialogue_id":  dialogue_id,
            "dataset_type": dataset_type,
            "category":     category,
            "model":        model,
            "condition":    condition_key,
            "speaker":      speaker,
            "rating":       None,
            "explanation":  f"ERROR: {e}",
            "status":       "failed",
        }


print("run_rating defined.")

run_rating defined.


Data Frame Helpers

In [28]:

COLUMNS = [
    "dialogue_id", "dataset_type", "category",
    "model", "condition", "speaker",
    "rating", "explanation", "status",
]


def load_or_create_df(csv_path: Path) -> pd.DataFrame:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        logger.info("Loaded existing CSV with %d rows from %s", len(df), csv_path)
        return df
    logger.info("No existing CSV found at %s. Starting fresh.", csv_path)
    return pd.DataFrame(columns=COLUMNS)


def is_completed(
    df: pd.DataFrame,
    dialogue_id: int,
    model: str,
    condition: str,
    speaker: str,
) -> bool:
    if df.empty:
        return False
    mask = (
        (df["dialogue_id"] == dialogue_id)
        & (df["model"] == model)
        & (df["condition"] == condition)
        & (df["speaker"] == speaker)
        & (df["status"] == "completed")
    )
    return bool(mask.any())


print("DataFrame helpers defined.")

DataFrame helpers defined.


Main Run

In [ ]:

def main(
    dataset_type: str = DATASET_TYPE,
    category:     str = CATEGORY,
    max_workers:  int = 8,
) -> pd.DataFrame:
    """Run ratings for the configured dataset/category. Returns the final DataFrame."""

    df = load_or_create_df(OUTPUT_CSV)

    tasks = []
    for idx, entry in enumerate(dialogues):
        dialogue_text = format_dialogue(entry)
        for condition_key in RUN_CONDITIONS:
            for model in MODELS:
                for speaker in ["Speaker 1", "Speaker 2"]:
                    if is_completed(df, idx, model, condition_key, speaker):
                        logger.info(
                            "Skipping completed: dialogue=%d model=%s condition=%s speaker=%s",
                            idx, model, condition_key, speaker,
                        )
                        continue
                    tasks.append((idx, dialogue_text, model, condition_key, speaker))

    logger.info("%d tasks to run (dataset=%s, category=%s).", len(tasks), dataset_type, category)

    if not tasks:
        logger.info("All tasks already completed. Nothing to do.")
        return df

    def worker(task):
        d_id, d_text, model, cond, spkr = task
        return run_rating(d_id, d_text, model, cond, spkr, dataset_type, category)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for result in executor.map(worker, tasks):
            new_row = pd.DataFrame([result])
            df = pd.concat([df, new_row], ignore_index=True)
            df.to_csv(OUTPUT_CSV, index=False)  

    logger.info("Done. %d total rows saved to %s", len(df), OUTPUT_CSV)
    return df


# Run!
results_df = main()
results_df.tail()

INFO: No existing CSV found at C:\Users\AL MAKKAH\Desktop\vsc\llm-bias\multi_modal_analysis_categorized\figurative_analysis\semi-transparent.csv. Starting fresh.
INFO: 1920 tasks to run (dataset=figurative, category=semi-transparent).
ERROR: run_rating failed (model=azure-deepseek-r1, condition=ctx_all, speaker=Speaker 2): No valid JSON found in response: <think>
Okay, let's tackle this. The user wants me to analyze the dialogue between Speaker 1 and Speaker 2, determine which speaker is autistic, and evaluate Speaker 2's communication from a neurodive
Traceback (most recent call last):
  File "C:\Users\AL MAKKAH\AppData\Local\Temp\ipykernel_15488\2458798410.py", line 36, in run_rating
    raise ValueError(f"No valid JSON found in response: {raw[:200]}")
ValueError: No valid JSON found in response: <think>
Okay, let's tackle this. The user wants me to analyze the dialogue between Speaker 1 and Speaker 2, determine which speaker is autistic, and evaluate Speaker 2's communication from a

,dialogue_id,dataset_type,category,model,condition,speaker,rating,explanation,status
1915,39,figurative,semi-transparent,gpt3-5,ctx_autistic_s2,Speaker 2,2,Speaker 2 takes the idiomatic expression 'jump...,completed
1916,39,figurative,semi-transparent,azure-llama3.3-70b,ctx_autistic_s2,Speaker 1,5,Speaker 1 uses the idiomatic expression 'jumpe...,completed
1917,39,figurative,semi-transparent,azure-llama3.3-70b,ctx_autistic_s2,Speaker 2,5,Speaker 2's communication in this exchange app...,completed
1918,39,figurative,semi-transparent,azure-deepseek-r1,ctx_autistic_s2,Speaker 1,4,Speaker 1 used the idiom 'jumped on the bandwa...,completed
1919,39,figurative,semi-transparent,azure-deepseek-r1,ctx_autistic_s2,Speaker 2,3,Speaker 2's response ('Do we need music instru...,completed


Rerun Failed Rows

In [ ]:

def rerun_failed() -> None:
    if not OUTPUT_CSV.exists():
        logger.info("No CSV found. Run main() first.")
        return

    df = pd.read_csv(OUTPUT_CSV)
    failed = df[df["status"] == "failed"]
    logger.info("%d failed rows to rerun.", len(failed))

    if failed.empty:
        logger.info("No failed rows. Nothing to rerun.")
        return

    tasks = []
    for _, row in failed.iterrows():
        idx = int(row["dialogue_id"])
        if idx >= len(dialogues):
            logger.warning("dialogue_id %d out of range — skipping.", idx)
            continue
        dialogue_text = format_dialogue(dialogues[idx])
        tasks.append((idx, dialogue_text, row["model"], row["condition"], row["speaker"]))

    def worker(task):
        d_id, d_text, model, cond, spkr = task
        return run_rating(d_id, d_text, model, cond, spkr, DATASET_TYPE, CATEGORY)

    with ThreadPoolExecutor(max_workers=8) as executor:
        for result in executor.map(worker, tasks):
            mask = (
                (df["dialogue_id"] == result["dialogue_id"])
                & (df["model"] == result["model"])
                & (df["condition"] == result["condition"])
                & (df["speaker"] == result["speaker"])
                & (df["status"] == "failed")
            )
            df.loc[mask, ["rating", "explanation", "status"]] = [
                result["rating"], result["explanation"], result["status"]
            ]
            df.to_csv(OUTPUT_CSV, index=False)

    logger.info("Rerun complete. CSV updated at %s", OUTPUT_CSV)


rerun_failed()

INFO: 0 failed rows to rerun.
INFO: No failed rows. Nothing to rerun.


Quick Summary

In [ ]:


if OUTPUT_CSV.exists():
    summary_df = pd.read_csv(OUTPUT_CSV)
    print(f"CSV: {OUTPUT_CSV}")
    print(f"Total rows : {len(summary_df)}")
    print(f"Completed  : {(summary_df['status'] == 'completed').sum()}")
    print(f"Failed     : {(summary_df['status'] == 'failed').sum()}")
    print()
    print("Mean rating by condition and speaker:")
    pivot = (
        summary_df[summary_df["status"] == "completed"]
        .groupby(["condition", "speaker"])["rating"]
        .mean()
        .unstack("speaker")
        .round(2)
    )
    print(pivot)
else:
    print("No CSV yet — run Cell 8 first.")

CSV: C:\Users\AL MAKKAH\Desktop\vsc\llm-bias\multi_modal_analysis_categorized\figurative_analysis\semi-transparent.csv
Total rows : 1920
Completed  : 1920
Failed     : 0

Mean rating by condition and speaker:
speaker          Speaker 1  Speaker 2
condition                            
ctx_all               2.73       2.65
ctx_autism            2.71       3.66
ctx_autistic_s1       1.96       4.01
ctx_autistic_s2       3.90       2.94
ctx_neuro             2.84       3.19
ctx_none              2.48       3.47
